In [ ]:
# %%
# Buoc 1a — Cai dat thu vien

!pip -q uninstall -y numpy
!pip -q install numpy==1.26.4 mlflow scikit-learn xgboost fastapi "uvicorn[standard]" pydantic requests

# Tai cloudflared binary (Cloudflare Tunnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
     -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('Cai dat xong! → Chay cell Buoc 1b ben duoi de RESTART RUNTIME')


In [ ]:
# Buoc 1b — RESTART RUNTIME (bat buoc sau khi cai numpy moi)
# Runtime se restart — sau do bat dau tu Buoc 2
import os
os.kill(os.getpid(), 9)


In [ ]:
# %%
# Tao cau truc thu muc project
import os

PROJECT = "/content/ngohongthong"  # Doi ten theo ten ban (khong dau)

DIRS = [
    f"{PROJECT}/Data_Pipeline",
    f"{PROJECT}/Model_Training_Tracking",
    f"{PROJECT}/Model_Versioning_Registry",
    f"{PROJECT}/Model_Deployment",
    f"{PROJECT}/CICD_Monitoring",
]

for d in DIRS:
    os.makedirs(d, exist_ok=True)
    print(f"Created: {d}")

print("\nCau truc thu muc da san sang!")


In [ ]:
# %%
# Buoc 2 — Data Pipeline: Lam sach + TF-IDF + Luu dataset

import pandas as pd
import numpy as np
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from google.colab import files

PROJECT = "/content/ngohongthong"

# === 1. Upload dataset ===
print("Upload file spamDataset.csv:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# === 2. Doc va lam sach ===
df = pd.read_csv(filename, encoding='latin-1')
print(f"\nShape ban dau: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Chi giu 2 cot chinh
df = df[['v1', 'v2']].copy()
df.columns = ['label', 'text']

# Xoa dong trong / trung lap
df.dropna(subset=['text'], inplace=True)
df.drop_duplicates(inplace=True)

# Chuyen label thanh so: ham=0, spam=1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Lam sach text
def clean_text(text):
    text = str(text).lower()                        # lowercase
    text = re.sub(r'http\S+|www\.\S+', '', text)    # xoa URL
    text = re.sub(r'[^a-zA-Z\s]', '', text)         # xoa ky tu dac biet
    text = re.sub(r'\s+', ' ', text).strip()         # xoa khoang trang thua
    return text

df['text_clean'] = df['text'].apply(clean_text)

print(f"Shape sau lam sach: {df.shape}")
print(f"Phan bo label:\n{df['label'].value_counts()}")
print(f"\nMau du lieu:")
print(df[['label', 'text_clean']].head(10).to_string(index=False))

# === 3. Split du lieu: train / val / test (60/20/20) ===
X = df['text_clean'].values
y = df['label_num'].values

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# === 4. TF-IDF ===
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print(f"TF-IDF shape: {X_train_tfidf.shape}")

# === 5. Luu dataset da xu ly ===
DATA_DIR = f"{PROJECT}/Data_Pipeline"

df.to_csv(f"{DATA_DIR}/spam_cleaned.csv", index=False)

with open(f"{DATA_DIR}/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

data_bundle = {
    "X_train": X_train_tfidf, "X_val": X_val_tfidf, "X_test": X_test_tfidf,
    "y_train": y_train,       "y_val": y_val,       "y_test": y_test,
    "X_train_raw": X_train,   "X_val_raw": X_val,   "X_test_raw": X_test,
}
with open(f"{DATA_DIR}/data_splits.pkl", "wb") as f:
    pickle.dump(data_bundle, f)

print(f"\nDa luu: spam_cleaned.csv, tfidf_vectorizer.pkl, data_splits.pkl")
print(f"Thu muc: {DATA_DIR}")


In [ ]:
# %% [markdown]
# ## Buoc 3 — Model Training & Tracking (MLflow)
#
# Huan luyen 2 mo hinh: **Naive Bayes** va **XGBoost**
#
# Log vao MLflow: Parameters, Metrics (Precision, Recall, F1), Model Artifacts


In [ ]:
# %%
# Cau hinh MLflow
import os
import mlflow

PROJECT = "/content/ngohongthong"

MLFLOW_DB        = "sqlite:////content/mlflow_spam.db"
MLFLOW_ARTIFACTS = "/content/mlartifacts_spam"
os.makedirs(MLFLOW_ARTIFACTS, exist_ok=True)

mlflow.set_tracking_uri(MLFLOW_DB)

EXPERIMENT_NAME = "Spam_Classification"
mlflow.set_experiment(EXPERIMENT_NAME)

os.environ["GUNICORN_CMD_ARGS"] = "--forwarded-allow-ips='*'"

print(f"Tracking URI  : {mlflow.get_tracking_uri()}")
print(f"Experiment    : {EXPERIMENT_NAME}")
print(f"MLflow version: {mlflow.__version__}")


In [ ]:
# %%
# Buoc 3 — Huan luyen Naive Bayes + XGBoost, log vao MLflow

import pickle
import numpy as np
import mlflow.sklearn
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)
from mlflow.models import infer_signature

PROJECT  = "/content/ngohongthong"
DATA_DIR = f"{PROJECT}/Data_Pipeline"

# Load du lieu da xu ly
with open(f"{DATA_DIR}/data_splits.pkl", "rb") as f:
    data = pickle.load(f)

X_train = data["X_train"]
X_val   = data["X_val"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_val   = data["y_val"]
y_test  = data["y_test"]

with open(f"{DATA_DIR}/tfidf_vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)

# Dinh nghia 2 mo hinh
models = {
    "NaiveBayes": {
        "model": MultinomialNB(alpha=1.0),
        "params": {"algorithm": "MultinomialNB", "alpha": 1.0},
    },
    "XGBoost": {
        "model": XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=42,
        ),
        "params": {
            "algorithm": "XGBoost",
            "n_estimators": 200,
            "max_depth": 6,
            "learning_rate": 0.1,
        },
    },
}

run_ids = {}

for name, cfg in models.items():
    with mlflow.start_run(run_name=name) as run:
        model = cfg["model"]
        model.fit(X_train, y_train)

        # Predict tren Validation
        y_val_pred = model.predict(X_val)
        val_prec = precision_score(y_val, y_val_pred)
        val_rec  = recall_score(y_val, y_val_pred)
        val_f1   = f1_score(y_val, y_val_pred)
        val_acc  = accuracy_score(y_val, y_val_pred)

        # Predict tren Test
        y_test_pred = model.predict(X_test)
        test_prec = precision_score(y_test, y_test_pred)
        test_rec  = recall_score(y_test, y_test_pred)
        test_f1   = f1_score(y_test, y_test_pred)
        test_acc  = accuracy_score(y_test, y_test_pred)

        # Log params
        mlflow.log_params(cfg["params"])
        mlflow.log_params({
            "tfidf_max_features": 5000,
            "train_size": X_train.shape[0],
            "val_size":   X_val.shape[0],
            "test_size":  X_test.shape[0],
        })

        # Log metrics
        mlflow.log_metrics({
            "val_accuracy":  val_acc,
            "val_precision": val_prec,
            "val_recall":    val_rec,
            "val_f1":        val_f1,
            "test_accuracy":  test_acc,
            "test_precision": test_prec,
            "test_recall":    test_rec,
            "test_f1":        test_f1,
        })

        # Log model artifact
        signature = infer_signature(X_test, y_test_pred)
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="spam_model",
            registered_model_name="SpamClassifier",
            signature=signature,
        )

        run_ids[name] = run.info.run_id

        print(f"\n{'='*50}")
        print(f"Model: {name} | Run ID: {run.info.run_id[:8]}")
        print(f"  Val  → Prec={val_prec:.4f} | Rec={val_rec:.4f} | F1={val_f1:.4f}")
        print(f"  Test → Prec={test_prec:.4f} | Rec={test_rec:.4f} | F1={test_f1:.4f}")
        print(f"\nClassification Report (Test):")
        print(classification_report(y_test, y_test_pred, target_names=["ham", "spam"]))

print("\nDa log xong tat ca runs vao MLflow!")
print(f"Run IDs: {run_ids}")


In [ ]:
# %% [markdown]
# ## Buoc 4 — Model Versioning & Registry
#
# - Dang ky model tot nhat vao Model Registry
# - Gan stage: Staging va Production
# - So sanh 2 version model


In [ ]:
# %%
# Buoc 4 — Model Versioning & Registry

import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd

mlflow.set_tracking_uri("sqlite:////content/mlflow_spam.db")
client = MlflowClient()

PROJECT = "/content/ngohongthong"

# === 1. Lay tat ca runs, sap xep theo test_f1 ===
exp = client.get_experiment_by_name("Spam_Classification")
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.test_f1 DESC"],
)

print("=== So sanh 2 mo hinh ===")
print(f"{'Model':<15} {'Test Prec':>10} {'Test Rec':>10} {'Test F1':>10} {'Test Acc':>10}")
print("-" * 58)

for r in runs:
    print(
        f"{r.info.run_name:<15}"
        f" {r.data.metrics.get('test_precision',0):>10.4f}"
        f" {r.data.metrics.get('test_recall',0):>10.4f}"
        f" {r.data.metrics.get('test_f1',0):>10.4f}"
        f" {r.data.metrics.get('test_accuracy',0):>10.4f}"
    )

best_run = runs[0]
second_run = runs[1] if len(runs) > 1 else None

print(f"\nModel tot nhat: {best_run.info.run_name} (test_f1 = {best_run.data.metrics['test_f1']:.4f})")

# === 2. Lay cac versions trong Registry ===
versions = client.search_model_versions("name='SpamClassifier'")
versions_sorted = sorted(versions, key=lambda v: int(v.version))

print(f"\nModel Registry — SpamClassifier: {len(versions_sorted)} versions")

# === 3. Gan Staging cho version dau, Production cho version tot nhat ===
_use_alias = True

# Tim version tuong ung voi best run va second run
best_version = None
staging_version = None

for v in versions_sorted:
    if v.run_id == best_run.info.run_id:
        best_version = v.version
    elif second_run and v.run_id == second_run.info.run_id:
        staging_version = v.version

# Gan alias/stage
try:
    if staging_version:
        client.set_registered_model_alias("SpamClassifier", "Staging", staging_version)
        print(f"\nVersion {staging_version} → alias 'Staging'")

    if best_version:
        client.set_registered_model_alias("SpamClassifier", "Production", best_version)
        print(f"Version {best_version} → alias 'Production'")

except AttributeError:
    _use_alias = False
    if staging_version:
        client.transition_model_version_stage(
            name="SpamClassifier", version=staging_version,
            stage="Staging", archive_existing_versions=False,
        )
        print(f"\nVersion {staging_version} → stage 'Staging'")

    if best_version:
        client.transition_model_version_stage(
            name="SpamClassifier", version=best_version,
            stage="Production", archive_existing_versions=False,
        )
        print(f"Version {best_version} → stage 'Production'")

# Tag
client.set_registered_model_tag("SpamClassifier", "task", "spam_classification")
client.set_model_version_tag("SpamClassifier", best_version, "test_f1",
                              str(round(best_run.data.metrics["test_f1"], 4)))

# === 4. Hien thi Registry ===
print(f"\n{'='*60}")
print("Model Registry — SpamClassifier:")
for v in client.search_model_versions("name='SpamClassifier'"):
    aliases = getattr(v, "aliases", [])
    stage   = getattr(v, "current_stage", "N/A")
    run_name = ""
    for r in runs:
        if r.info.run_id == v.run_id:
            run_name = r.info.run_name
            break
    print(f"  Version {v.version} | {run_name:<15} | Stage: {stage} | Aliases: {aliases} | Run: {v.run_id[:8]}")

# === 5. Giai thich vi sao chon model nay ===
print(f"\n{'='*60}")
print("GIAI THICH: Vi sao chon model tot nhat?")
print("-" * 60)
if best_run and second_run:
    b = best_run.data.metrics
    s = second_run.data.metrics
    print(f"  {best_run.info.run_name}:")
    print(f"    Test F1={b['test_f1']:.4f}, Precision={b['test_precision']:.4f}, Recall={b['test_recall']:.4f}")
    print(f"  {second_run.info.run_name}:")
    print(f"    Test F1={s['test_f1']:.4f}, Precision={s['test_precision']:.4f}, Recall={s['test_recall']:.4f}")
    print(f"\n  → {best_run.info.run_name} co F1 cao hon, can bang tot giua Precision va Recall.")
    print(f"    Trong bai toan spam, Recall cao giup khong bo sot spam,")
    print(f"    Precision cao giup khong phan loai nham ham thanh spam.")
    print(f"    Model nay phu hop de deploy len Production.")

BEST_MODEL_VERSION = best_version
print(f"\nModel version cho Production: {BEST_MODEL_VERSION}")

# Luu thong tin de cell sau dung
import json
registry_info = {
    "best_version": best_version,
    "staging_version": staging_version,
    "best_run_name": best_run.info.run_name,
}
with open(f"{PROJECT}/Model_Versioning_Registry/registry_info.json", "w") as f:
    json.dump(registry_info, f)


In [ ]:
# %% [markdown]
# ## Buoc 5 — Model Deployment (FastAPI)
#
# - Load model tu MLflow Registry
# - Endpoint `/predict`: Input text string → Output predicted label
# - Load TF-IDF vectorizer de transform text moi


In [ ]:
# %%
%%writefile /content/prediction_api.py
import os
import re
import pickle
import numpy as np
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field
import mlflow.sklearn
from datetime import datetime
import json

os.environ["MLFLOW_TRACKING_URI"] = "sqlite:////content/mlflow_spam.db"

app = FastAPI(
    title="Spam Classification API",
    description="Phan loai email spam/ham — Model load tu MLflow Registry",
    version="1.0",
)

_model = None
_tfidf = None

# === Monitoring: Log prediction requests ===
LOG_FILE = "/content/ngohongthong/CICD_Monitoring/prediction_log.jsonl"
os.makedirs(os.path.dirname(LOG_FILE), exist_ok=True)


def log_prediction(text, prediction, confidence):
    entry = {
        "timestamp": datetime.now().isoformat(),
        "text_length": len(text),
        "word_count": len(text.split()),
        "prediction": prediction,
        "confidence": round(confidence, 4),
    }
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def get_tfidf():
    global _tfidf
    if _tfidf is None:
        tfidf_path = "/content/ngohongthong/Data_Pipeline/tfidf_vectorizer.pkl"
        with open(tfidf_path, "rb") as f:
            _tfidf = pickle.load(f)
        print(f"[API] TF-IDF vectorizer loaded")
    return _tfidf


def get_model():
    global _model
    if _model is not None:
        return _model

    uris_to_try = [
        "models:/SpamClassifier@Production",
        "models:/SpamClassifier/Production",
    ]
    for uri in uris_to_try:
        try:
            _model = mlflow.sklearn.load_model(uri)
            print(f"[API] Model loaded tu: {uri}")
            return _model
        except Exception:
            continue

    try:
        from mlflow.tracking import MlflowClient
        client = MlflowClient()
        versions = client.search_model_versions("name='SpamClassifier'")
        latest = sorted(versions, key=lambda v: int(v.version))[-1]
        uri = f"models:/SpamClassifier/{latest.version}"
        _model = mlflow.sklearn.load_model(uri)
        print(f"[API] Model loaded (fallback): {uri}")
        return _model
    except Exception as e:
        raise RuntimeError(f"Khong the load model: {e}")


class TextInput(BaseModel):
    text: str = Field(..., min_length=1, description="Noi dung email can phan loai")

    model_config = {
        "json_schema_extra": {
            "examples": [{
                "text": "Congratulations! You have won a free prize. Call now!"
            }]
        }
    }


class PredictResponse(BaseModel):
    prediction: str
    confidence: float
    label_id:   int


@app.get("/", response_class=HTMLResponse)
def root():
    return """
    <html><head><title>Spam Classification API</title>
    <style>
      body { font-family: sans-serif; max-width: 700px; margin: 3rem auto; padding: 1rem; }
      h1   { color: #2563eb; }
      .badge { background: #dcfce7; color: #15803d; padding: 2px 10px;
               border-radius: 99px; font-size: .85rem; }
      a    { color: #2563eb; }
      code { background: #f1f5f9; padding: 2px 6px; border-radius: 4px; }
    </style></head><body>
    <h1>Spam Classifier <span class='badge'>MLflow + FastAPI</span></h1>
    <p>Model load tu <strong>MLflow Model Registry</strong></p>
    <ul>
      <li><code>GET /health</code> — kiem tra trang thai</li>
      <li><code>POST /predict</code> — phan loai email (input: text string)</li>
      <li><a href='/docs'>GET /docs</a> — Swagger UI</li>
      <li><code>GET /drift</code> — kiem tra data drift</li>
    </ul>
    </body></html>"""


@app.get("/health")
def health():
    try:
        get_model()
        get_tfidf()
        return {"status": "ok", "model": "SpamClassifier"}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/predict", response_model=PredictResponse)
def predict(data: TextInput):
    model = get_model()
    tfidf = get_tfidf()

    text_clean = clean_text(data.text)
    X = tfidf.transform([text_clean])

    pred = int(model.predict(X)[0])
    label = "spam" if pred == 1 else "ham"

    # Lay confidence
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)[0]
        conf = float(proba[pred])
    else:
        conf = 1.0

    # Log prediction cho monitoring
    log_prediction(data.text, label, conf)

    return PredictResponse(
        prediction=label,
        confidence=round(conf, 4),
        label_id=pred,
    )


@app.get("/drift")
def drift_detection():
    """Detect data drift bang do dai trung binh cua email."""
    # Do dai trung binh cua tap train (tinh san)
    import pickle
    data_path = "/content/ngohongthong/Data_Pipeline/data_splits.pkl"
    with open(data_path, "rb") as f:
        d = pickle.load(f)
    train_lengths = [len(str(t)) for t in d["X_train_raw"]]
    train_avg = np.mean(train_lengths)
    train_std = np.std(train_lengths)

    # Do dai trung binh cua cac request gan day
    if not os.path.exists(LOG_FILE):
        return {"status": "no_data", "message": "Chua co prediction log"}

    recent_lengths = []
    with open(LOG_FILE, "r") as f:
        for line in f:
            entry = json.loads(line.strip())
            recent_lengths.append(entry["text_length"])

    if len(recent_lengths) < 5:
        return {"status": "insufficient_data", "count": len(recent_lengths)}

    recent_avg = np.mean(recent_lengths)
    drift_score = abs(recent_avg - train_avg) / (train_std + 1e-8)

    drift_detected = drift_score > 2.0  # > 2 std → drift

    return {
        "status": "drift_detected" if drift_detected else "no_drift",
        "train_avg_length": round(train_avg, 2),
        "recent_avg_length": round(recent_avg, 2),
        "drift_score": round(drift_score, 4),
        "threshold": 2.0,
        "total_predictions": len(recent_lengths),
        "recommendation": "Nen retrain model!" if drift_detected
                          else "Model van on dinh.",
    }


In [ ]:
# %% [markdown]
# ## Buoc 6 — Khoi dong Services + Cloudflare Tunnel
#
# Expose 2 services ra internet:
# - **Port 5000** → MLflow UI
# - **Port 8000** → Prediction API + Swagger Docs


In [ ]:
# %%
# Buoc 6 — Khoi dong MLflow Server + FastAPI + Cloudflare Tunnel

import subprocess, threading, time, re, urllib.request, urllib.error, os

MLFLOW_ARTIFACTS = "/content/mlartifacts_spam"
os.makedirs(MLFLOW_ARTIFACTS, exist_ok=True)

os.environ["MLFLOW_ENABLE_PROXY_FIX"] = "true"
os.environ["GUNICORN_CMD_ARGS"]        = "--forwarded-allow-ips='*'"

mlflow_url = {"url": None}
api_url    = {"url": None}


def stream_output(proc, prefix):
    for line in iter(proc.stdout.readline, b""):
        print(prefix + line.decode(errors="replace").rstrip())


def wait_for_port(port, timeout=60):
    start = time.time()
    while time.time() - start < timeout:
        try:
            urllib.request.urlopen(f"http://localhost:{port}/", timeout=2)
            return True
        except urllib.error.HTTPError:
            return True
        except Exception:
            time.sleep(1)
    return False


def cf_tunnel(port, holder, label):
    p = subprocess.Popen(
        [
            "/usr/local/bin/cloudflared", "tunnel",
            "--url", f"http://localhost:{port}",
            "--no-autoupdate",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    for line in iter(p.stdout.readline, b""):
        txt = line.decode(errors="replace").rstrip()
        m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", txt)
        if m and not holder["url"]:
            holder["url"] = m.group(0)
            print(f"\n{label}: {holder['url']}")


# 1. MLflow Server
p_mlflow = subprocess.Popen(
    [
        "mlflow", "server",
        "--host",                  "0.0.0.0",
        "--port",                  "5000",
        "--backend-store-uri",     "sqlite:////content/mlflow_spam.db",
        "--default-artifact-root", f"file://{MLFLOW_ARTIFACTS}",
        "--allowed-hosts",         "*",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env={**os.environ},
)
threading.Thread(target=stream_output, args=(p_mlflow, "[MLflow] "), daemon=True).start()

print("Cho MLflow server khoi dong...", end="", flush=True)
if wait_for_port(5000, timeout=60):
    print(" ready")
else:
    print(" timeout — xem log [MLflow] ben tren")

# 2. FastAPI
p_api = subprocess.Popen(
    [
        "python", "-m", "uvicorn", "prediction_api:app",
        "--host", "0.0.0.0",
        "--port", "8000",
    ],
    cwd="/content",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env={**os.environ},
)
threading.Thread(target=stream_output, args=(p_api, "[API]    "), daemon=True).start()

print("Cho FastAPI khoi dong...", end="", flush=True)
if wait_for_port(8000, timeout=40):
    print(" ready")
else:
    print(" timeout — xem log [API] ben tren")

# 3. Cloudflare Tunnels
time.sleep(2)
threading.Thread(target=cf_tunnel, args=(5000, mlflow_url, "MLflow UI"), daemon=True).start()

time.sleep(3)
threading.Thread(target=cf_tunnel, args=(8000, api_url, "Prediction API"), daemon=True).start()

print("\nDang lay Cloudflare URL (toi da ~60 giay)...")
for _ in range(40):
    time.sleep(2)
    if mlflow_url["url"] and api_url["url"]:
        break

print("\n" + "=" * 65)
print(f"MLflow UI    : {mlflow_url.get('url', 'chua co URL')}")
print(f"API          : {api_url.get('url',    'chua co URL')}")
if api_url.get("url"):
    print(f"Swagger Docs : {api_url['url']}/docs")
    print(f"Health Check : {api_url['url']}/health")
    print(f"Drift Check  : {api_url['url']}/drift")
print("=" * 65)


In [ ]:
# %% [markdown]
# ## Buoc 7 — Test Prediction API
#
# Test voi nhieu mau email spam va ham.


In [ ]:
# %%
# Buoc 7 — Test API

import requests, time

BASE = api_url.get("url", "http://localhost:8000")
print(f"Testing API tai: {BASE}\n")

# Cho API san sang
for _ in range(5):
    try:
        r = requests.get(f"{BASE}/health", timeout=10)
        if r.status_code == 200:
            print(f"Health: {r.json()}\n")
            break
    except Exception:
        time.sleep(3)

# Cac mau test
samples = [
    {"text": "Hey, are you coming to the party tonight? Let me know!"},
    {"text": "CONGRATULATIONS! You've won $1,000,000! Call now to claim your prize at 0800-WIN-NOW"},
    {"text": "Hi mom, I'll be home for dinner. Love you!"},
    {"text": "FREE entry to win a brand new iPhone! Text WIN to 80888. T&C apply."},
    {"text": "Can we reschedule the meeting to 3pm tomorrow?"},
    {"text": "URGENT: Your account has been compromised. Click here to verify: http://scam.com"},
    {"text": "Ok lar... Joking wif u oni..."},
    {"text": "You have been selected for a cash prize! Call 09061701461 to claim NOW"},
]

print("Ket qua Prediction API:")
print("-" * 65)
for s in samples:
    try:
        r   = requests.post(f"{BASE}/predict", json=s, timeout=20)
        res = r.json()
        tag = "SPAM" if res["prediction"] == "spam" else "HAM "
        print(f"[{tag}] (conf={res['confidence']:.2%}) | {s['text'][:70]}")
    except Exception as e:
        print(f"Loi: {e}")

print("\n" + "-" * 65)
print("Test XONG!")


In [ ]:
# %% [markdown]
# ## Buoc 8 — Monitoring & Drift Detection
#
# - Log prediction requests (da tich hop trong API)
# - Detect data drift don gian bang do dai trung binh cua email
# - De xuat khi nao nen retrain model


In [ ]:
# %%
# Buoc 8 — Monitoring: Xem prediction log + Drift detection

import requests, json, os
import pandas as pd

BASE = api_url.get("url", "http://localhost:8000")
PROJECT = "/content/ngohongthong"

# === 1. Xem prediction log ===
LOG_FILE = f"{PROJECT}/CICD_Monitoring/prediction_log.jsonl"

if os.path.exists(LOG_FILE):
    logs = []
    with open(LOG_FILE, "r") as f:
        for line in f:
            logs.append(json.loads(line.strip()))

    df_log = pd.DataFrame(logs)
    print("=== Prediction Log (gan nhat) ===")
    print(df_log.tail(10).to_string(index=False))
    print(f"\nTong so predictions: {len(df_log)}")
    print(f"Spam: {(df_log['prediction']=='spam').sum()} | Ham: {(df_log['prediction']=='ham').sum()}")
else:
    print("Chua co prediction log")

# === 2. Goi endpoint /drift ===
print(f"\n{'='*55}")
print("=== Data Drift Detection ===")
try:
    r = requests.get(f"{BASE}/drift", timeout=10)
    drift = r.json()
    print(json.dumps(drift, indent=2, ensure_ascii=False))

    if drift.get("status") == "drift_detected":
        print("\nCANH BAO: Data drift detected!")
        print("De xuat: Nen retrain model voi du lieu moi.")
    elif drift.get("status") == "no_drift":
        print("\nModel van on dinh, chua can retrain.")
except Exception as e:
    print(f"Loi: {e}")

# === 3. De xuat khi nao retrain ===
print(f"\n{'='*55}")
print("=== De xuat khi nao retrain model ===")
print("""
Nen retrain model khi:
  1. Drift score > 2.0 (do dai trung binh email thay doi qua nhieu so voi tap train)
  2. Accuracy/F1 giam tren du lieu thuc te (giam sat metric theo thoi gian)
  3. Phan bo spam/ham thay doi dang ke (vd: ti le spam tang dot bien)
  4. Sau 1-3 thang hoat dong, du lieu moi du lon de retrain
  5. Khi co loai spam moi ma model cu khong nhan dien duoc
""")


In [ ]:
# %% [markdown]
# ## Buoc 9 — Truy van Experiments & Model Registry bang Python


In [ ]:
# %%
# Buoc 9 — Tong hop Experiments & Registry

import mlflow, pandas as pd
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("sqlite:////content/mlflow_spam.db")
client = MlflowClient()

exp  = client.get_experiment_by_name("Spam_Classification")
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.test_f1 DESC"],
)

rows = []
for r in runs:
    rows.append({
        "run_id"    : r.info.run_id[:8],
        "model"     : r.info.run_name,
        "val_f1"    : round(r.data.metrics.get("val_f1", 0), 4),
        "test_f1"   : round(r.data.metrics.get("test_f1", 0), 4),
        "test_prec" : round(r.data.metrics.get("test_precision", 0), 4),
        "test_rec"  : round(r.data.metrics.get("test_recall", 0), 4),
        "test_acc"  : round(r.data.metrics.get("test_accuracy", 0), 4),
        "status"    : r.info.status,
    })

df = pd.DataFrame(rows)
print("=== Bang tong hop Runs (sorted by test_f1) ===")
print(df.to_string(index=False))

print("\n=== Model Registry — SpamClassifier ===")
for v in client.search_model_versions("name='SpamClassifier'"):
    aliases = getattr(v, "aliases", [])
    stage   = getattr(v, "current_stage", "N/A")
    run_name = ""
    for r in runs:
        if r.info.run_id == v.run_id:
            run_name = r.info.run_name
            break
    print(f"  Version {v.version} | {run_name:<15} | Stage: {stage} | Aliases: {aliases}")

print(f"\nMLflow UI: {mlflow_url.get('url', 'N/A')}")
print(f"API:       {api_url.get('url', 'N/A')}")


## Bước 10 — CI/CD Pipeline (GitHub Actions)

Yêu cầu 5 của đề: viết file `.github/workflows/mlops.yml` để tự động hóa pipeline khi push code lên GitHub.

**Pipeline gồm 3 step:**
1. **Run data pipeline** — chạy lại data cleaning + TF-IDF
2. **Train model** — huấn luyện 2 model + log MLflow
3. **Run test** — chạy unit test cho monitoring/drift detection


In [ ]:
# %%
# Buoc 10 — Tao file CI/CD GitHub Actions
# File nay se duoc dat tai .github/workflows/mlops.yml trong repo GitHub
# Khi push code len branch main, GitHub Actions se tu dong chay pipeline

import os

# Tao thu muc theo dung chuan GitHub Actions
PROJECT  = "/content/ngohongthong"
CICD_DIR = f"{PROJECT}/CICD_Monitoring/.github/workflows"
os.makedirs(CICD_DIR, exist_ok=True)

# Noi dung file YAML — moi step la mot cong viec tu dong hoa
mlops_yaml = """name: MLOps Pipeline

# Trigger: chay khi push hoac mo PR vao branch main
on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  mlops-pipeline:
    runs-on: ubuntu-latest

    steps:
      # Step 1: Lay code ve runner
      - name: Checkout source code
        uses: actions/checkout@v4

      # Step 2: Cai Python 3.10 cho runner
      - name: Setup Python 3.10
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'

      # Step 3: Cai cac thu vien can thiet
      - name: Cai dat dependencies
        run: |
          python -m pip install --upgrade pip
          pip install pandas scikit-learn xgboost mlflow fastapi uvicorn pydantic joblib jupyter nbconvert pytest requests

      # Step 4: Chay Data Pipeline (lam sach + TF-IDF + luu dataset)
      - name: Run Data Pipeline
        run: jupyter nbconvert --to notebook --execute Data_Pipeline/data_pipeline.ipynb

      # Step 5: Train model + log MLflow
      - name: Train Model
        run: jupyter nbconvert --to notebook --execute Model_Training_Tracking/train.ipynb

      # Step 6: Chay unit test (kiem tra drift detection)
      - name: Run Tests
        run: pytest CICD_Monitoring/tests/ -v
"""

# Ghi file
yaml_path = f"{CICD_DIR}/mlops.yml"
with open(yaml_path, "w") as f:
    f.write(mlops_yaml)

print(f"Da tao file CI/CD tai: {yaml_path}")
print(f"
Noi dung file:
{'='*60}")
print(mlops_yaml)


### Tạo unit test cho drift detection

File test này sẽ được chạy ở step **Run Tests** trong CI/CD pipeline.


In [ ]:
# %%
# Buoc 10b — Tao unit test cho drift detection
# CI/CD se chay file test nay de dam bao logic drift detection van dung

import os

PROJECT  = "/content/ngohongthong"
TEST_DIR = f"{PROJECT}/CICD_Monitoring/tests"
os.makedirs(TEST_DIR, exist_ok=True)

# Tao file __init__.py rong de pytest nhan ra package
with open(f"{TEST_DIR}/__init__.py", "w") as f:
    f.write("")

# Noi dung file test
test_code = """\"\"\"Unit test don gian cho drift detection.\"\"\"
import numpy as np


def detect_drift(recent_lengths, baseline_mean, baseline_std, threshold=2.0):
    \"\"\"Phat hien drift dua tren do dai trung binh email.

    - Neu chenh lech > threshold * std => co drift
    - Tra ve True neu phat hien drift, False neu khong.
    \"\"\"
    if not recent_lengths:
        return False
    recent_mean = float(np.mean(recent_lengths))
    diff = abs(recent_mean - baseline_mean)
    return diff > threshold * baseline_std


def test_no_drift_khi_du_lieu_giong_baseline():
    # Du lieu moi co do dai gan giong baseline => khong drift
    assert detect_drift([95, 100, 105, 102, 98],
                        baseline_mean=100, baseline_std=20) is False


def test_co_drift_khi_du_lieu_lech_xa():
    # Du lieu moi lech xa baseline => phat hien drift
    assert detect_drift([300, 320, 310, 305, 315],
                        baseline_mean=100, baseline_std=20) is True


def test_input_rong_khong_bao_drift():
    # Khong co du lieu => khong the ket luan => False
    assert detect_drift([], baseline_mean=100, baseline_std=20) is False
"""

test_path = f"{TEST_DIR}/test_drift.py"
with open(test_path, "w") as f:
    f.write(test_code)

print(f"Da tao file test tai: {test_path}")

# Chay thu test luon de kiem tra
print(f"
{'='*60}")
print("Chay thu unit test:")
import subprocess
result = subprocess.run(
    ["python", "-m", "pytest", test_path, "-v"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


## Bước 11 — Tổng kết & Hướng dẫn nộp bài

Sau khi chạy xong toàn bộ notebook, bạn có:
- ✅ **Yêu cầu 1** — Data Pipeline (TF-IDF + clean text)
- ✅ **Yêu cầu 2** — Training & Tracking (NB + XGBoost + MLflow)
- ✅ **Yêu cầu 3** — Model Registry (Staging + Production + so sánh)
- ✅ **Yêu cầu 4** — FastAPI `/predict` (load model từ Registry + test)
- ✅ **Yêu cầu 5** — CI/CD + Monitoring (drift detection + retrain logic)

### Cần chụp ảnh để nộp:
1. **MLflow UI** — vào URL ở Bước 6, tab Experiments → chụp 2 run NB và XGBoost
2. **Swagger Docs** — `<api_url>/docs` → chụp endpoint `/predict`
3. **Test API** — output của Bước 7 (8 mẫu test) hoặc dùng curl/Postman


In [ ]:
# %%
# Buoc 11 — Liet ke toan bo file da tao trong project (de check truoc khi nop)

import os

PROJECT = "/content/ngohongthong"

print(f"Cau truc thu muc cua project '{os.path.basename(PROJECT)}':")
print("=" * 60)

for root, dirs, files in os.walk(PROJECT):
    # Bo qua thu muc an
    dirs[:] = [d for d in dirs if not d.startswith(".__")]
    level  = root.replace(PROJECT, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(root, f))
        print(f"{indent}  {f}  ({size:,} bytes)")

print("\n" + "=" * 60)
print("Hoan tat! Kiem tra dam bao co du cac file:")
print("  Data_Pipeline/spam_cleaned.csv, tfidf_vectorizer.pkl, data_splits.pkl")
print("  Model_Versioning_Registry/registry_info.json")
print("  CICD_Monitoring/prediction_log.jsonl")
print("  CICD_Monitoring/.github/workflows/mlops.yml")
print("  CICD_Monitoring/tests/test_drift.py")
